In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print("Device:", device)

In [ ]:
raw_tf = transforms.ToTensor()
cifar = datasets.CIFAR10(root="./data", train=True, download=True, transform=raw_tf)
img_tensor, label = cifar[0]          
img_input = img_tensor.unsqueeze(0)   
print("Image shape:", img_input.shape)

In [ ]:
def make_conv(weight_2d):
    conv = nn.Conv2d(3, 1, kernel_size=3, padding=1, bias=False)
    w = torch.tensor(weight_2d, dtype=torch.float32)
    conv.weight.data = w.unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1)
    return conv

vertical = [[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]]
horizontal = [[-1, -1, -1], [0, 0, 0], [1, 1, 1]]
blur = (np.ones((3, 3)) / 9).tolist()

filters = {"Vertical Edge": vertical, "Horizontal Edge": horizontal, "Blur": blur}

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(img_tensor.permute(1, 2, 0).numpy())
axes[0].set_title("Original")

with torch.no_grad():
    for ax, (name, filt) in zip(axes[1:], filters.items()):
        out = make_conv(filt)(img_input)
        ax.imshow(out.squeeze().numpy(), cmap="gray")
        ax.set_title(name)

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2)

x = torch.randn(8, 3, 32, 32)
print("Input:      ", x.shape)

model = TinyCNN()
x1 = model.conv1(x);  print("After conv1:", x1.shape)
x2 = model.pool1(x1); print("After pool1:", x2.shape)
x3 = model.conv2(x2); print("After conv2:", x3.shape)
x4 = model.pool2(x3); print("After pool2:", x4.shape)

### Part A — Filter observations
- **Vertical Edge**: Highlights vertical transitions in pixel intensity (left-to-right contrast). Edges of objects running top-to-bottom become visible.
- **Horizontal Edge**: Highlights horizontal transitions (top-to-bottom contrast). Edges running left-to-right are enhanced.
- **Blur**: Averages each pixel with its neighbours, smoothing out sharp details and reducing noise.

### Part B — Shape Tracking Table

| Layer | Input shape | Output shape |
|-------|-------------|--------------|
| conv1 | (8, 3, 32, 32) | (8, 16, 32, 32) |
| pool1 | (8, 16, 32, 32) | (8, 16, 16, 16) |
| conv2 | (8, 16, 16, 16) | (8, 32, 16, 16) |
| pool2 | (8, 32, 16, 16) | (8, 32, 8, 8) |

### Task 2

In [ ]:
base_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

train_set = datasets.CIFAR10(root="./data", train=True,  download=False, transform=base_tf)
val_set   = datasets.CIFAR10(root="./data", train=False, download=False, transform=base_tf)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=128, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
class CIFARCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),  nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x)

model = CIFARCNN().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(model, loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(training):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct   += (outputs.argmax(1) == labels).sum().item()
            total     += labels.size(0)
    return total_loss / total, correct / total

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, 16):
    tr_loss, tr_acc = run_epoch(model, train_loader, training=True)
    vl_loss, vl_acc = run_epoch(model, val_loader,   training=False)
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)
    print(f"Epoch {epoch:2d} | Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f} | Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}")

In [ ]:
epochs = range(1, 16)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, history["train_loss"], label="Train Loss")
ax1.plot(epochs, history["val_loss"],   label="Val Loss")
ax1.set_title("Loss — No Augmentation")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(epochs, history["train_acc"], label="Train Acc")
ax2.plot(epochs, history["val_acc"],   label="Val Acc")
ax2.set_title("Accuracy — No Augmentation")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\nBest Val Accuracy (no augmentation): {max(history['val_acc']):.4f}")
print(f"Train/Val gap at best val epoch:     {history['train_acc'][np.argmax(history['val_acc'])] - max(history['val_acc']):.4f}")

Trainable parameters: 1,117,354

The model converges steadily over 15 epochs.

Final best validation accuracy: 79.98% (exceeds the expected 70–75% baseline).

Train/Val gap at best epoch: 0.0877 — mild overfitting begins in later epochs as train accuracy climbs above val accuracy.

### Task 3

In [ ]:
train_tf_aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

train_set_aug = datasets.CIFAR10(root="./data", train=True,  download=False, transform=train_tf_aug)
train_loader_aug = DataLoader(train_set_aug, batch_size=128, shuffle=True, num_workers=0)

# Re-initialise a fresh model and optimizer
model_aug = CIFARCNN().to(device)
optimizer_aug = optim.Adam(model_aug.parameters(), lr=1e-3)

def run_epoch_aug(model, loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(training):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if training:
                optimizer_aug.zero_grad()
                loss.backward()
                optimizer_aug.step()
            total_loss += loss.item() * labels.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, correct / total

history_aug = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, 16):
    tr_loss, tr_acc = run_epoch_aug(model_aug, train_loader_aug, training=True)
    vl_loss, vl_acc = run_epoch_aug(model_aug, val_loader,       training=False)
    history_aug["train_loss"].append(tr_loss)
    history_aug["val_loss"].append(vl_loss)
    history_aug["train_acc"].append(tr_acc)
    history_aug["val_acc"].append(vl_acc)
    print(f"Epoch {epoch:2d} | Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f} | Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, history_aug["train_loss"], label="Train Loss (aug)")
ax1.plot(epochs, history_aug["val_loss"],   label="Val Loss (aug)")
ax1.set_title("Loss — With Augmentation")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(epochs, history_aug["train_acc"], label="Train Acc (aug)")
ax2.plot(epochs, history_aug["val_acc"],   label="Val Acc (aug)")
ax2.set_title("Accuracy — With Augmentation")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()

best_aug_val = max(history_aug["val_acc"])
best_aug_gap = history_aug["train_acc"][np.argmax(history_aug["val_acc"])] - best_aug_val
print(f"\nBest Val Accuracy (with augmentation): {best_aug_val:.4f}")
print(f"Train/Val gap at best val epoch:       {best_aug_gap:.4f}")

In [ ]:
best_base_val = max(history["val_acc"])
best_base_gap = history["train_acc"][np.argmax(history["val_acc"])] - best_base_val

print(f"{'Run':<35} {'Best Val Acc':>12} {'Train/Val Gap':>14}")
print("-" * 63)
print(f"{'Task 2 (no augmentation)':<35} {best_base_val:>12.4f} {best_base_gap:>14.4f}")
print(f"{'Task 3 (with augmentation)':<35} {best_aug_val:>12.4f} {best_aug_gap:>14.4f}")

| Run | Best Val Accuracy | Train/Val Gap |
|-----|-------------------|---------------|
| Task 2 (no augmentation) | 0.7998 | 0.0877 |
| Task 3 (with augmentation) | 0.8042 | -0.0312 |

### Observations :

Val accuracy improved slightly (79.98% → 80.42%) with augmentation, despite the augmented model training on harder, randomly transformed images. This shows augmentation helps generalisation even within 15 epochs.

The train/val gap shrank dramatically — from +0.0877 down to -0.0312. The negative gap means validation accuracy actually *exceeded* training accuracy, which is a hallmark of effective regularisation: augmentation makes the training set harder than the clean validation set, so the model is never overconfident on training data.

Training accuracy was lower with augmentation (77.30% vs 88.75%) because the model sees a different random crop/flip/jitter each epoch, making memorisation harder. This is exactly the intended effect.

In summary: data augmentation acts as a strong regulariser — it reduces overfitting, narrows the generalisation gap, and yields a model that performs better on unseen data, at the cost of slower convergence during training. This effect would become even more pronounced with more epochs.